In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import wandb
import pandas as pd
from transformers import Trainer, Seq2SeqTrainer, Seq2SeqTrainingArguments

import os
from datetime import datetime

os.chdir("../scripts")

from data_processing import poquad, processing
from t5.load_t5 import *

In [3]:
train_df, valid_df = poquad.load_poquad_manually_downloaded("../data/poquad-manually-processed/")

In [4]:
train_input = poquad.dataset_into_str_input(train_df)

In [5]:
valid_input = poquad.dataset_into_str_input(valid_df)

In [6]:
tokenizer, model = load_plt5("../models/plt5-original-small")

In [7]:
from transformers import TrainerCallback

class PrintMemoryUsageCallback(TrainerCallback):
    """ Callback that prints memory allocation during training """
    def on_step_end(self, args, state, control, **kwargs):
        print(f"Step {state.global_step}: {torch.cuda.memory_allocated() / 1024 ** 2:.2f} MB allocated")

In [8]:
wandb.init(
    # set the wandb project where this run will be logged
    project="PLT5 Small Finetuning Poquad",
    # track hyperparameters and run metadata
    # config={
    # "architecture": "PLT5 Small",
    # "dataset": "Poquad",
    # }
)


train_dataset = processing.TextDataset(train_input, tokenizer, 1024, 128)
valid_dataset = processing.TextDataset(valid_input, tokenizer, 1024, 128)

num_train_samples = train_dataset.__len__()
train_batch_size = 5
gradient_accumulation_steps = 1

num_train_steps_per_epoch = (num_train_samples // train_batch_size // gradient_accumulation_steps) + 1

# Calculate save steps for every 2 epochs
save_steps = 2 * num_train_steps_per_epoch

# Initialize model

# Define TrainingArguments
training_args = Seq2SeqTrainingArguments(
    learning_rate=3e-4,
    output_dir='./results',
    run_name=f'plt5-small-{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}',
    num_train_epochs=8,
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    save_total_limit=4,
    warmup_steps=500,
    weight_decay=0.01,
    gradient_accumulation_steps=gradient_accumulation_steps,  # Gradient accumulation steps
    logging_dir='./logs',
    overwrite_output_dir=True,
    save_steps=save_steps,  # Save checkpoint every 2 epochs
    metric_for_best_model="eval_loss",  # Use evaluation loss to determine the best model
    greater_is_better=False,  # Lower eval_loss is better
    logging_steps=100,  # Log every 100 steps
    predict_with_generate=True,  # Use generate method for predictions
)


# Initialize Trainer
trainer = Seq2SeqTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        # data_collator=data_collator,
        callbacks=[PrintMemoryUsageCallback()],
        )

# # Train the model
trainer.train()

# Save the model
# trainer.save_model('../models/plt5-small-1epoch-overfit')

# print("Training complete and model saved")
wandb.finish()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: m-tkacz6 (m-tkacz-uw). Use `wandb login --relogin` to force relogin


  0%|          | 0/90592 [00:00<?, ?it/s]

Step 1: 1107.59 MB allocated
Step 2: 1107.59 MB allocated
Step 3: 1107.59 MB allocated
Step 4: 1107.59 MB allocated
Step 5: 1107.59 MB allocated
Step 6: 1107.59 MB allocated
Step 7: 1107.59 MB allocated
Step 8: 1107.59 MB allocated
Step 9: 1107.59 MB allocated
Step 10: 1107.59 MB allocated
Step 11: 1107.59 MB allocated
Step 12: 1107.59 MB allocated
Step 13: 1107.59 MB allocated
Step 14: 1107.59 MB allocated
Step 15: 1107.59 MB allocated
Step 16: 1107.59 MB allocated
Step 17: 1107.59 MB allocated
Step 18: 1107.59 MB allocated
Step 19: 1107.59 MB allocated
Step 20: 1107.59 MB allocated
Step 21: 1107.59 MB allocated
Step 22: 1107.59 MB allocated
Step 23: 1107.59 MB allocated
Step 24: 1107.59 MB allocated
Step 25: 1107.59 MB allocated
Step 26: 1107.59 MB allocated
Step 27: 1107.59 MB allocated
Step 28: 1107.59 MB allocated
Step 29: 1107.59 MB allocated
Step 30: 1107.59 MB allocated
Step 31: 1107.59 MB allocated
Step 32: 1107.59 MB allocated
Step 33: 1107.59 MB allocated
Step 34: 1107.59 MB

train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,█▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
train/loss,█▅▅▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,1.938621182383227e+17
train/epoch,8.0
train/global_step,90592
train/grad_norm,0.13754
train/learning_rate,0.0
train/loss,0.0159
